# Gravità: un modello troppo grande

Il peso è **F = m × g**: una retta, che un solo neurone impara con 2 numeri (vedi 02a_gravita_neurone).
Qui la facciamo imparare a una rete con 4 strati da 512 neuroni.
Guarda quanti numeri usa e quanto è precisa.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers, models

# Constants - accelerazione gravitazionale
g = 9.8  # gravity on Earth in m/s^2

# Generate synthetic data
def generate_data(num_samples=1000):
    m = np.random.uniform(1, 1000, num_samples)  # Mass in kg
    F = m * g  # Force (weight) in Newtons
    return m, F

# Generate training and testing data
m_train, F_train = generate_data(800)
m_test, F_test = generate_data(200)

# Reshape the data
m_train = m_train.reshape(-1, 1)  # TensorFlow expects input as (batch_size, num_features)
m_test = m_test.reshape(-1, 1)

# Build the model
'''
model = models.Sequential([
    layers.Input(shape=(1,)), # Input layer
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)  # Output layer for the force
])
'''
model = models.Sequential([
       layers.Input(shape=(1,)),
       layers.Dense(512, activation='relu'),
       layers.Dense(512, activation='relu'),
       layers.Dense(512, activation='relu'),
       layers.Dense(512, activation='relu'),  # More layers and neurons
       layers.Dense(1)
   ])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model
history = model.fit(m_train, F_train, epochs=5, batch_size=32, validation_data=(m_test, F_test))

# Evaluate the model
loss = model.evaluate(m_test, F_test)
print(f"Test Loss: {loss}")

# Make predictions
predictions = model.predict(m_test)



Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 9s 69ms/step - loss: 19702628.0000 - val_loss: 1199415.2500
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - loss: 435823.4062 - val_loss: 181225.4062
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - loss: 63141.9805 - val_loss: 14351.5566
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 34ms/step - loss: 5808.3989 - val_loss: 251.6112
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - loss: 219.0893 - val_loss: 80.5659
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 77.2886 
Test Loss: 80.56591033935547
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step


# Plot it

In [ ]:

# Compare predictions to actual values
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.scatter(F_test, predictions.flatten(), alpha=0.5, color='#1F3A5F')
plt.plot([min(F_test), max(F_test)], [min(F_test), max(F_test)], color='#C25A00', linestyle='--')
plt.xlabel("Actual Force (N)")
plt.ylabel("Predicted Force (N)")
plt.title("Actual vs Predicted Force (Weight)")
plt.show()

In [ ]:
# Create a Pandas DataFrame for the table
import pandas as pd

predictions_flat = predictions.flatten() # Flatten predictions before subtraction

# Calcola le differenze
differences =  F_test - predictions_flat

 # Flatten predictions to a 1D array
predictions = predictions.flatten()

df = pd.DataFrame({
    'Valori attesi': F_test,
    'Valore predetto': predictions,
    "Delta": differences     # Flatten differences as well
})

# Display the table
print(df)

     Valori attesi  Valore predetto      Delta
0      3206.308758      3212.054688  -5.745930
1      3555.690987      3561.898926  -6.207939
2      6804.560787      6815.065918 -10.505131
3      2869.638856      2874.938965  -5.300109
4      2549.519688      2554.396484  -4.876797
..             ...              ...        ...
195    5531.353241      5540.174805  -8.821563
196    8668.358590      8681.329102 -12.970512
197    2563.718504      2568.614258  -4.895754
198    9459.805368      9473.823242 -14.017875
199    1140.779662      1143.793091  -3.013429

[200 rows x 3 columns]


In [ ]:
# Una rete a più strati non ha un "g" da leggere: i suoi pesi non sono la formula.
print(f'Numeri da imparare: {model.count_params():,} (il neurone singolo ne usa 2)')

# Proviamo anche masse fuori dai dati di addestramento (che andavano da 1 a 1000 kg).
masse = np.array([[10.0], [500.0], [1000.0], [2000.0], [5000.0]])
previsti = model.predict(masse, verbose=0).flatten()
print(pd.DataFrame({'Massa (kg)': masse.flatten(), 'Forza vera (N)': masse.flatten() * g,
                    'Forza prevista (N)': previsti.round(1)}))

La rete arriva vicino alla risposta, ma sbaglia sempre un po', anche dentro i dati che conosce,
e per farlo usa centinaia di migliaia di numeri. Il neurone singolo trova 9.8 esatto con 2.
Un modello più grande non è un modello migliore: serve quello adatto al problema.

In [ ]:
# Get the number of layers and neurons
num_layers = len(model.layers) - 1  # Exclude the input layer
neurons_per_layer = [layer.units for layer in model.layers[1:]]  # Exclude input layer

# Plot training and validation loss
plt.figure(figsize=(12, 8))  # Increase figure size for better visibility

# Plot training loss
plt.plot(history.history['loss'], label='Training Loss', color='#1F3A5F', linewidth=2)

# Plot validation loss
plt.plot(history.history['val_loss'], label='Validation Loss', color='#C25A00', linewidth=2, linestyle='--')

# Customize plot elements
plt.xlabel('Epoch', fontsize=14)
plt.ylabel('Loss', fontsize=14)
plt.title(f'Training and Validation Loss ({num_layers} Layers, {neurons_per_layer} Neurons)', fontsize=16)  # Include model details in the title

plt.legend(fontsize=12)

# Add a grid for better readability
plt.grid(True)

# Find the epoch where validation loss starts increasing consistently
overfitting_epoch = np.argmin(history.history['val_loss']) + 5

# Make sure overfitting_epoch is within the valid range
overfitting_epoch = min(overfitting_epoch, len(history.history['val_loss']) - 1)

# Add a vertical line and annotation to highlight overfitting
plt.axvline(x=overfitting_epoch, color='gray', linestyle='dotted', linewidth=2)

# Dynamically adjust annotation position to avoid overlap with loss curve
y_annotation = max(history.history['val_loss'][overfitting_epoch],
                    history.history['loss'][overfitting_epoch]) + 0.1

plt.annotate('Overfitting Starts',
            xy=(overfitting_epoch, y_annotation),
            xytext=(overfitting_epoch + 5, y_annotation + 0.1),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=12, color='gray')

plt.show()